In [1]:
tabla='cmsho10'
dim='dim_servicios'

In [2]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD_2"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [3]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMSHO10", con=connection2)

connection2.close()




In [4]:
base2

,servhoscod,servhosdes,servhosdescor,servhossolcitvol,indsexadmcod,servhosedaadmmin,servhosedaadmmax,tipopardiacod,estregcod,servhostipatecod,servhospotpagate,servhostipprocod,servhosevalpqxflg,servhosespqxflg,servhoseqvssaludcod,servhosgensolopeflg,servhosriecarflg,servhosrieneuflg,servhosinfanesflg
0,AA1,MEDICINA FISICA Y REHABILITACION,MED.FIS.Y REHABIL.,1,2,0,999,01,1,2,1,1,0,0,UPSS19,None,None,None,None
1,AB1,MEDICINA GENERAL,MEDICINA GENERAL,1,2,14,999,01,1,2,1,1,0,0,UPSS26,None,None,None,None
2,AC1,MEDICINA INTERNA,MEDICINA INTERNA,1,2,0,999,01,1,2,1,1,0,0,UPSS27,0,1,0,None
3,AD1,NEFROLOGIA,NEFROLOGIA,1,2,0,999,01,1,2,1,1,0,1,UPSS31,1,0,0,None
4,AF1,NEUROLOGIA,NEUROLOGIA,1,2,0,999,01,1,2,1,1,0,0,UPSS35,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,L12,EVALUACION MEDICA DE LA INCAPACIDAD,EVAL.MED.DE INCAPAC.,1,2,18,999,01,1,2,1,1,0,0,,None,None,None,None
194,A89,"HEMATOLOGÍA, HEMOTERAPIA Y BANCO DE SANGRE",HEMAT.HEMOT.BCO.SAN.,0,2,0,999,01,1,2,1,1,0,0,None,None,None,None,None
195,FB9,BIOQUIMICA E INMUNOQUIMICA,BIOQUIM.E INMUNOQUIM,0,2,0,999,00,1,0,1,1,0,0,None,None,None,None,None
196,AL1,EPIDEMIOLOGIA,EPIDEMIOLOGIA,1,2,0,999,01,1,2,1,1,0,0,UPSS28,None,None,None,None


In [5]:
base2.columns

Index(['servhoscod', 'servhosdes', 'servhosdescor', 'servhossolcitvol',
       'indsexadmcod', 'servhosedaadmmin', 'servhosedaadmmax', 'tipopardiacod',
       'estregcod', 'servhostipatecod', 'servhospotpagate', 'servhostipprocod',
       'servhosevalpqxflg', 'servhosespqxflg', 'servhoseqvssaludcod',
       'servhosgensolopeflg', 'servhosriecarflg', 'servhosrieneuflg',
       'servhosinfanesflg'],
      dtype='object')

In [6]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM cmsho10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cmsho10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cmsho10 ()')
base2.to_sql(name='tmp_cmsho10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmsho10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cmsho10 
ALTER COLUMN servhoscod TYPE character(3),
ALTER COLUMN servhosdes TYPE character(50),
ALTER COLUMN servhosdescor TYPE character(20),
ALTER COLUMN servhossolcitvol TYPE character(1),
ALTER COLUMN indsexadmcod TYPE character(1),
ALTER COLUMN servhosedaadmmin TYPE numeric(3,0),
ALTER COLUMN servhosedaadmmax TYPE numeric(3,0),
ALTER COLUMN tipopardiacod TYPE character(2),
ALTER COLUMN estregcod TYPE character(1),
ALTER COLUMN servhostipatecod TYPE character(1),
ALTER COLUMN servhospotpagate TYPE character(1),
ALTER COLUMN servhostipprocod TYPE character(1),
ALTER COLUMN servhosevalpqxflg TYPE character(1),
ALTER COLUMN servhosespqxflg TYPE character(1),
ALTER COLUMN servhoseqvssaludcod TYPE character(7);




UPDATE cmsho10 
SET 
servhosdes = t.servhosdes,
servhosdescor = t.servhosdescor,
servhossolcitvol = t.servhossolcitvol,
indsexadmcod = t.indsexadmcod,
servhosedaadmmin = t.servhosedaadmmin,
servhosedaadmmax = t.servhosedaadmmax,
tipopardiacod = t.tipopardiacod,
estregcod = t.estregcod,
servhostipatecod = t.servhostipatecod,
servhospotpagate = t.servhospotpagate,
servhostipprocod = t.servhostipprocod,
servhosevalpqxflg = t.servhosevalpqxflg,
servhosespqxflg = t.servhosespqxflg,
servhoseqvssaludcod = t.servhoseqvssaludcod   

FROM tmp_cmsho10 t 
WHERE cmsho10.servhoscod = t.servhoscod and cmsho10.servhoscod IS NOT NULL;

INSERT INTO cmsho10 
(servhoscod,servhosdes,servhosdescor,servhossolcitvol,indsexadmcod,servhosedaadmmin,servhosedaadmmax,tipopardiacod,estregcod,servhostipatecod,servhospotpagate,servhostipprocod,servhosevalpqxflg,servhosespqxflg,servhoseqvssaludcod) 
SELECT 
servhoscod,servhosdes,servhosdescor,servhossolcitvol,indsexadmcod,servhosedaadmmin,servhosedaadmmax,tipopardiacod,estregcod,servhostipatecod,servhospotpagate,servhostipprocod,servhosevalpqxflg,servhosespqxflg,servhoseqvssaludcod

FROM tmp_cmsho10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmsho10 
    WHERE cmsho10.servhoscod = t.servhoscod and cmsho10.servhoscod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmsho10;
DELETE FROM cmsho10 WHERE servhoscod ='';
SELECT COUNT(*) FROM cmsho10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbtae10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cmsho10", con=connection3)









connection3.close()


Cantidad de filas en la tabla cmsho10 antes de la inserción: 198
Cantidad de filas en la tabla mbtae10 después de la inserción: 198
La cantidad de filas insertadas fue de: 0


In [7]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN servhoscod TYPE character(3),
ALTER COLUMN servhosdes TYPE character(50),
ALTER COLUMN servhosdescor TYPE character(20),
ALTER COLUMN servhossolcitvol TYPE character(1),
ALTER COLUMN indsexadmcod TYPE character(1),
ALTER COLUMN servhosedaadmmin TYPE numeric(3,0),
ALTER COLUMN servhosedaadmmax TYPE numeric(3,0),
ALTER COLUMN tipopardiacod TYPE character(2),
ALTER COLUMN estregcod TYPE character(1),
ALTER COLUMN servhostipatecod TYPE character(1),
ALTER COLUMN servhospotpagate TYPE character(1),
ALTER COLUMN servhostipprocod TYPE character(1),
ALTER COLUMN servhosevalpqxflg TYPE character(1),
ALTER COLUMN servhosespqxflg TYPE character(1),
ALTER COLUMN servhoseqvssaludcod TYPE character(7);
INSERT INTO {dim} 

(cod_ser,des_ser,abr_ser) 

SELECT 


servhoscod,servhosdes,servhosdescor

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_ser = t.servhoscod
);
"""

c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_servicios antes de la inserción: 0
Cantidad de filas en la tabla dim_servicios después de la inserción: 198
La cantidad de filas insertadas fue de: 198
